# BinSense — M3: Labeling (single-class object detection)

Produce YOLO-format boxes (`data/labels/`) for the detector (M4) via the
three-stage pipeline: **manual seed (Label Studio) → SAM-assisted → zero-shot**.
One class: `item`. Box every **visible** unit (metadata is a cross-check, not a
target). Full convention: **docs/03_LABELING_GUIDE.md**. Eval-gold bins are held
out of *every* stage.

> Colab-first for the GPU stages (SAM / zero-shot); the manual-ingest and
> dataset-build stages also run locally.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split
import sys, os, subprocess
from pathlib import Path

GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'opencv-python-headless', 'tqdm', 'pandas', 'pyyaml'], check=True)
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Running in  : {"Google Colab" if IN_COLAB else "Local"}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA DIR    : {os.getenv("BINSENSE_DATA_DIR", str(PROJECT_ROOT / "data"))}')
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"

In [ ]:
# Cell 2: Imports + paths
import json, subprocess
from pathlib import Path
import pandas as pd
from utils.env_utils import setup_env

cfg = setup_env(verbose=True)
LABELS_DIR = cfg.labels_dir                     # YOLO .txt exports land here
IMAGES_DIR = cfg.images_dir
META_DIR   = cfg.metadata_dir
SPLITS_DIR = cfg.splits_dir
STAGING    = cfg.data_dir / 'labels_staging'    # auto-label output for HUMAN REVIEW
STAGING.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

# The one wall we never cross: eval-gold bins are the M7 test set.
EVAL_IDS = (set(pd.read_csv(SPLITS_DIR / 'eval.csv')['bin_id'].astype(str).str.zfill(5))
            if (SPLITS_DIR / 'eval.csv').exists() else set())
print('labels :', LABELS_DIR)
print('staging:', STAGING)
print('eval bins held out:', len(EVAL_IDS))

## Step 1 — Manual seed via Label Studio

The 120-bin manual seed is the ground-truth template for the whole detector.
Tasks are already generated (eval-gold excluded, leak-free). Regenerate only if
needed, then label per **docs/03_LABELING_GUIDE.md** and export YOLO → `data/labels/`.

In [ ]:
# Cell 3: Confirm (or regenerate) Label Studio tasks + eval-leakage guard
TASKS_JSON = cfg.base_dir / 'data' / 'label_studio_tasks.json'
REGENERATE = False   # set True to re-draw the 120-bin sample

if REGENERATE:
    subprocess.run([sys.executable,
                    str(cfg.base_dir / 'tools' / 'labeling' / 'make_label_studio_tasks.py')],
                   check=True)

tasks = json.loads(TASKS_JSON.read_text(encoding='utf-8'))
task_ids = {str(t['data']['bin_id']).zfill(5) for t in tasks}
leak = task_ids & EVAL_IDS
assert not leak, f'EVAL LEAKAGE in tasks: {sorted(leak)}'
print(f'{len(tasks)} tasks — eval-leakage check OK (0 eval bins).')
print('Config: tools/labeling/label_config.xml   Guide: docs/03_LABELING_GUIDE.md')

## Step 2 — Ingest exported manual labels + QA

After exporting YOLO from Label Studio into `data/labels/`, summarize coverage and
**overlay boxes back on the images** to catch coordinate-mapping bugs before
training. A gap between box-count and `EXPECTED_QUANTITY` is normal — we label
visible instances.

In [ ]:
# Cell 4: Summarize labels + render overlay QA
from tools.labeling.overlay_check import summarize, draw_overlay, parse_label

if not any(LABELS_DIR.glob('*.txt')):
    print('No YOLO labels in', LABELS_DIR,
          '\nExport from Label Studio first (see docs/03_LABELING_GUIDE.md).')
else:
    rows = summarize(LABELS_DIR, IMAGES_DIR, META_DIR)
    df = pd.DataFrame(rows)
    display(df[['bin_id', 'n_boxes', 'expected', 'delta']].head(20))

    out = cfg.base_dir / 'reports' / 'label_overlays'
    sample = df[df['img_exists']].sample(min(24, len(df)), random_state=42)
    for bid in sample['bin_id']:
        draw_overlay(IMAGES_DIR / f'{bid}.jpg',
                     parse_label(LABELS_DIR / f'{bid}.txt'),
                     out / f'{bid}.jpg')
    print(f'Wrote {len(sample)} overlays to {out} — open a few; boxes must hug items.')

## Step 3 — SAM-assisted labeling (Colab GPU · experimental)

Scale past the manual seed: run Segment Anything on unlabeled **extend** bins,
convert masks → boxes, stage for **human review** before merging. Never touch eval
bins. Area filters drop bin-sized and speck masks. `RUN_SAM=False` by default.

In [ ]:
# Cell 5: SAM auto-mask -> boxes (staging)   [experimental; GPU + checkpoint]
RUN_SAM  = False
SAM_N    = 25
SAM_TYPE = 'vit_b'
SAM_CKPT = 'sam_vit_b_01ec64.pth'   # place under models/ (download separately)

if RUN_SAM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segment-anything'], check=True)
    import cv2
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    from utils.env_utils import get_device

    sam = sam_model_registry[SAM_TYPE](checkpoint=str(cfg.models_dir / SAM_CKPT)).to(get_device())
    mg = SamAutomaticMaskGenerator(sam, points_per_side=32, pred_iou_thresh=0.86,
                                   stability_score_thresh=0.92, min_mask_region_area=500)

    extend = pd.read_csv(SPLITS_DIR / 'extend.csv')['bin_id'].astype(str).str.zfill(5).tolist()
    todo = [b for b in extend
            if b not in EVAL_IDS
            and not (LABELS_DIR / f'{b}.txt').exists()
            and not (STAGING / f'{b}.txt').exists()][:SAM_N]

    for bid in todo:
        img = cv2.imread(str(IMAGES_DIR / f'{bid}.jpg'))
        if img is None:
            continue
        H, W = img.shape[:2]
        lines_out = []
        for m in mg.generate(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)):
            x, y, w, h = m['bbox']                      # xywh px
            if w * h < 0.002 * W * H or w * h > 0.6 * W * H:   # drop specks / bin-sized
                continue
            lines_out.append(f'0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}')
        (STAGING / f'{bid}.txt').write_text('\n'.join(lines_out))
    print(f'SAM staged {len(todo)} bins -> {STAGING}  (REVIEW before merging into data/labels/)')
else:
    print('RUN_SAM=False — scaffold only. Enable on a Colab GPU runtime.')

## Step 4 — Zero-shot auto-label (YOLO-World · experimental)

Auto-box the remaining bulk with an open-vocabulary detector prompted for generic
objects, confidence-filter, stage for review. (GroundingDINO is an alternative;
decide the labeler in M3.) `RUN_ZEROSHOT=False` by default.

In [ ]:
# Cell 6: Zero-shot boxes via YOLO-World (staging)   [experimental]
RUN_ZEROSHOT = False
ZS_N       = 25
ZS_CONF    = 0.05
ZS_PROMPTS = ['product', 'box', 'package', 'item', 'bottle', 'bag']

if RUN_ZEROSHOT:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLOWorld
    model = YOLOWorld('yolov8x-worldv2.pt')
    model.set_classes(ZS_PROMPTS)

    extend = pd.read_csv(SPLITS_DIR / 'extend.csv')['bin_id'].astype(str).str.zfill(5).tolist()
    todo = [b for b in extend
            if b not in EVAL_IDS
            and not (LABELS_DIR / f'{b}.txt').exists()
            and not (STAGING / f'{b}.txt').exists()][:ZS_N]

    for bid in todo:
        r = model.predict(str(IMAGES_DIR / f'{bid}.jpg'), conf=ZS_CONF, verbose=False)[0]
        lines_out = [f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
                     for cx, cy, w, h in r.boxes.xywhn.tolist()]
        (STAGING / f'{bid}.txt').write_text('\n'.join(lines_out))
    print(f'Zero-shot staged {len(todo)} bins -> {STAGING}  (REVIEW before merging)')
else:
    print('RUN_ZEROSHOT=False — scaffold only.')

## Step 5 — Build the YOLO dataset (merge labels, EXCLUDE eval)

Assemble train/val from labeled bins and write `data.yaml`. Ultralytics finds each
label by swapping `/images/` → `/labels/` in the image path, so our
`data/images` + `data/labels` layout works directly. **Eval-gold bins are held out
entirely** — asserted, not assumed.

In [ ]:
# Cell 7: Assemble YOLO dataset manifest (train/val); assert no eval leakage
import yaml, random

labeled = sorted(p.stem for p in LABELS_DIR.glob('*.txt'))
labeled = [b for b in labeled if b not in EVAL_IDS]        # safety net
assert not (set(labeled) & EVAL_IDS), 'EVAL LEAKAGE in training labels!'

if not labeled:
    print('No labels yet — export the manual seed first, then re-run.')
else:
    random.Random(42).shuffle(labeled)
    n_val = max(1, int(0.15 * len(labeled)))
    val, train = labeled[:n_val], labeled[n_val:]

    YOLO_DIR = cfg.base_dir / 'data' / 'yolo'
    YOLO_DIR.mkdir(parents=True, exist_ok=True)
    for name, ids in [('train.txt', train), ('val.txt', val)]:
        (YOLO_DIR / name).write_text('\n'.join(str(IMAGES_DIR / f'{b}.jpg') for b in ids))

    (YOLO_DIR / 'data.yaml').write_text(yaml.safe_dump({
        'path': str(cfg.data_dir),
        'train': str(YOLO_DIR / 'train.txt'),
        'val': str(YOLO_DIR / 'val.txt'),
        'nc': 1, 'names': {0: 'item'},
    }, sort_keys=False))
    print(f'train={len(train)}  val={len(val)}  eval(held out)={len(EVAL_IDS)}')
    print('Wrote', YOLO_DIR / 'data.yaml')

## Step 6 — Coverage summary & handoff to M4

In [ ]:
# Cell 8: Coverage summary
n_labeled = len(list(LABELS_DIR.glob('*.txt')))
n_staged  = len(list(STAGING.glob('*.txt')))
print(f'Labeled (data/labels)   : {n_labeled}')
print(f'Staged for review       : {n_staged}')
print(f'Eval gold (held out)    : {len(EVAL_IDS)}')
print('\nNext: review staged auto-labels -> merge into data/labels/ -> M4 (train YOLO on data/yolo/data.yaml).')